In [1]:
from scipy.optimize import minimize
from scipy.optimize import direct, Bounds
from smooth_POD_ROM.post_processing import richardson_lucy
from smooth_POD_ROM.pre_processing import smoothen_rowwise
from smooth_POD_ROM.initial_conditions import rectangular_pulse, rect_pulse_sin, saw_tooth, triangle, small_saw_tooth, small_r_pulse
from smooth_POD_ROM.reduced_order_model import optimize_hyperparameters, optimize_sROM, optimize_sROMs_naive, target_function, get_predictions
from smooth_POD_ROM.reduced_order_model import make_data_rw, train_ROM_rw, L2_error_rw, get_data_rw, L1_error_rw
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

In [2]:
import matplotlib.pyplot as plt
import matplotlib as mpl
# \textwidth = 5.125in = 13.02
full_width = 130.2  # mm
single_column = 130.2/2  # mm
font_size_normal = 6

single_column_in = single_column/10/2.54
single_column_figure = (single_column_in, single_column_in/1.618)

plt.rcParams['font.size'] = font_size_normal
plt.rcParams["figure.figsize"] = single_column_figure

plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['legend.labelspacing'] = 0.25
plt.rcParams['legend.handlelength'] = 1.5
plt.rc('legend', fontsize=font_size_normal-2)
mpl.rc('text', usetex=True)
mpl.rc('font', family='serif', size=font_size_normal, serif='Computer Modern Roman')
pth = "../Plots/"

markevery = 40

In [3]:
def very_small_r_pulse(x, mu):
    return rectangular_pulse(x, mu, w=0.01)

def medium_r_pulse(x, mu):
    return rectangular_pulse(x, mu, w=0.02)

def step(x, mu, w=0.075 + 1e-6, periodic=True):
    y = np.zeros_like(x, dtype=np.float64)
    y[(x > mu)] = 1.0
    return y


def small_amp_pulse(x, mu):
    return rectangular_pulse(x, mu, w=0.075 + 1e-6)/100

n_x = 1000
x = np.linspace(0, 1, n_x, endpoint=False)
case = {
    "n_x": n_x,
    "shape": x.shape,
    "x": x,
    "dx": x[1] - x[0],

    "n_train": 50,
    "n_test": 25,  # 25
    "g": small_saw_tooth, #small_saw_tooth, #saw_tooth, #rectangular_pulse,step
    "truncate": 12,
    "sigma": 0.08,

    "c": 1,
    "monitor_progress_postprocessing": False,
    "num_iter": 100,
    "monitor_convergence": False,
    "sROM_only": False,
    "sigmaD": "calc_based_on_distance",
}

small_saw_tooth 0.02435185, 2.16049383
saw_tooth 0.01916255, 0.67901235
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
very_small_r_pulse 100, 30, 100, 0.00918313, 6.72839506, 0.10056346, 0.08570277, 0.03382032, -66.3692
rectangular_pulse  100, 30, 100, 0.01357407, 1.66666667, 0.09962107, 0.09608731, 0.05000187, -49.8079
step               100, 30, 100, 0.01397325, 1.17283951, 0.07018225, 0.09078748, 0.04862724, -30.7129

In [ ]:
NN = np.array([*np.arange(4, 24, 2), *np.arange(25, 75, 5), *np.arange(80, 200, 10), *np.arange(200, 500, 25)])  # only for medium pulse!
e_rom = np.zeros(len(NN),)
e_srom = np.zeros(len(NN),)
e_sroms0 = np.zeros(len(NN),)
e_sroms1 = np.zeros(len(NN),)
e_sroms2 = np.zeros(len(NN),)
sS_srom = np.zeros(len(NN),)
sS_sroms0 = np.zeros(len(NN),)
sD_sroms0 = np.zeros(len(NN),)
sS_sroms1 = np.zeros(len(NN),)
sD_sroms1 = np.zeros(len(NN),)
sS_sroms2 = np.zeros(len(NN),)
c_sroms2 = np.zeros(len(NN),)
for name, ic in zip(["medium_r_pulse", "small_amp_pulse", "step", "rect_pulse_sin", "triangle"], # "small_saw_tooth", "saw_tooth", "rectangular_pulse", "very_small_r_pulse", 
                    [medium_r_pulse, small_amp_pulse, step, rect_pulse_sin, triangle]): # , small_saw_tooth, saw_tooth, rectangular_pulse, very_small_r_pulse
    case["g"] = ic
    for i, rank in enumerate(NN):
        n_train = rank
        print(rank)
        case["rank"] = rank
        case["n_train"] = n_train
        
        case["sigmaD"] = "calc_based_on_distance"
        # S-ROM
        case["sROM_only"] = True
        case["n_test"] = 25
        parameters = optimize_sROM(case)
        case["n_test"] = 100
        case["sigma"] = parameters[0]
        # mean_ROM, mean_sROM, mean_sROMs = target_function(case)
        # e_rom[i] = mean_ROM
        # e_srom[i] = mean_sROM
        # sS_srom[i] = parameters[0]
        print("\n\n")

        # DS-ROM, sD=sS
        case["sROM_only"] = False
        case["n_test"] = 25
        parameters = optimize_sROMs_naive(case)
        case["sigmaD"] = np.zeros((100,))  # sigma_D given for each n_test
        case["n_test"] = 100
        case["sigma"], case["sigmaD"][:] = parameters[0], parameters[0]
        mean_ROM, mean_sROM, mean_sROMs = target_function(case)
        e_sroms0[i] = mean_sROMs
        sS_sroms0[i] = parameters[0]
        sD_sroms0[i] = parameters[1]
        print("\n\n")

        # DS-ROM, sD=const., but optimized
        case["sigma"], case["sigmaD"][:] = parameters[0], parameters[1]
        mean_ROM, mean_sROM, mean_sROMs = target_function(case)
        e_sroms1[i] = mean_sROMs
        sS_sroms1[i] = parameters[0]
        sD_sroms1[i] = parameters[1]
        print("\n\n")

        # DS-ROM, rule of thumb
        case["sROM_only"] = False
        case["sigmaD"] = "calc_based_on_distance"
        case["n_test"] = 25
        parameters = optimize_hyperparameters(case)
        case["n_test"] = 100
        case["sigma"], case["c"] = parameters[0], parameters[1]
        mean_ROM, mean_sROM, mean_sROMs = target_function(case)
        e_rom[i] = mean_ROM
        
        e_sroms2[i] = mean_sROMs
        sS_sroms2[i] = parameters[0]
        c_sroms2[i] = parameters[1]
        print("\n\n")
    
    pth = "../data/10_"+name

    np.save(pth+"_NN.npy", NN)

    np.save(pth+"_e_rom.npy", e_rom)
    
    np.save(pth+"_sS_srom.npy", sS_srom)
    np.save(pth+"_e_srom.npy", e_srom)
    
    np.save(pth+"10_sS_sroms0.npy", sS_sroms1)
    np.save(pth+"10_sD_sroms0.npy", sD_sroms1)
    np.save(pth+"_e_sroms0.npy", e_sroms1)
    
    np.save(pth+"10_sS_sroms1.npy", sS_sroms1)
    np.save(pth+"10_sD_sroms1.npy", sD_sroms1)
    np.save(pth+"_e_sroms1.npy", e_sroms1)
    
    np.save(pth+"10_sS_sroms2.npy", sS_sroms2)
    np.save(pth+"10_c_sroms2.npy", c_sroms2)
    np.save(pth+"_e_sroms2.npy", e_sroms2)
    N_max = 200
    NN = np.array([*np.arange(4, 24, 2), *np.arange(25, 75, 5), *np.arange(80, 200, 10)])  # now we cut the rank, as ICs are narrow


4
max: 0.05
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.02500500, -, 0.17038362, 0.14149005, -, -16.9580
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.04166833, -, 0.17038362, 0.13832214, -, -18.8172
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.00834167, -, 0.17038362, 0.15462795, -, -9.2472
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.04722278, -, 0.17038362, 0.13783334, -, -19.1041
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.03611389, -, 0.17038362, 0.13901527, -, -18.4104
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.03055944, -, 0.17038362, 0.14001203, -, -17.8254
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, me

C:\Repos\POD-ROM-and-gaussian-convolution\src\smooth_POD_ROM\post_processing.py:68: UserWarning: image needs to be 2D
  warnings.warn("image needs to be 2D")


num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.12550000, 0.12550000, 0.17038362, 0.13763800, 0.13576320, -20.3191
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.20850000, 0.12550000, 0.17038362, 0.13896183, 0.13839490, -18.7745
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.04250000, 0.12550000, 0.17038362, 0.13823771, 0.11827149, -30.5852
num_iter, n_train, n_test, sigma_S, sigma_D/c, mean_ROM, mean_sROM, mean_sROMs, improvement
100, 4, 25, 0.12550000, 0.20850000, 0.17038362, 0.13763800, 0.11824727, -30.5994


In [ ]:
case["sROM_only"] = False
case["sigmaD"] = "calc_based_on_distance"
case["num_iter"] = 100

# for name, ic in zip(["medium_r_pulse", "small_saw_tooth", "saw_tooth", "rectangular_pulse", "very_small_r_pulse", "step", "rect_pulse_sin", "triangle"],
#                     [medium_r_pulse, small_saw_tooth, saw_tooth, rectangular_pulse, very_small_r_pulse, step, rect_pulse_sin, triangle]):
for name, ic in zip(["rect_pulse_sin", "small_amp_pulse","saw_tooth","triangle","rectangular_pulse","medium_r_pulse","very_small_r_pulse","step","small_saw_tooth"],
                    [ rect_pulse_sin, small_amp_pulse, saw_tooth , triangle , rectangular_pulse , medium_r_pulse , very_small_r_pulse , step , small_saw_tooth ]):
    case["g"] = ic 
    pth = "../data/10_"+name
    NN = np.load(pth+"_NN.npy")
    
    sS_sroms2 = np.load(pth+"10_sS_sroms2.npy")
    c_sroms2 = np.load(pth+"10_c_sroms2.npy")
    # plot u_mu, u_rb, u_rb_DS for rank 10 and 100
    print(name)
    for rank in [10, 20, 100]:
        if rank not in NN:
            rank = rank-1
        i = np.where(NN==rank)[0][0]
        n_train = rank
        pth = "../data/10_"+name+"_r"+str(rank)
        print(rank)
        case["rank"] = rank
        case["n_train"] = n_train
        case["sigma"], case["c"] = sS_sroms2[i], c_sroms2[i]
        case["n_test"] = 4
        mu_train, X_train, X_train_s = make_data_rw([0], [1], n_samples=case["n_train"], **case)
        mu_test, X_test, X_test_s = make_data_rw(mu_train[n_train//4], mu_train[n_train//4+1], n_samples=case["n_test"], **case)
        my_ROM = train_ROM_rw(mu_train, X_train)
        my_sROM = train_ROM_rw(mu_train, X_train_s)
    
        X_test_ROM = my_ROM.predict(mu_test).snapshots_matrix
        X_test_sROM, X_test_sROMs = get_predictions(my_sROM, mu_test, **case)
        e_ROM = L2_error_rw(X_test_ROM, X_test)
        e_sROM = L2_error_rw(X_test_sROM, X_test)
        e_sROMs = L2_error_rw(X_test_sROMs, X_test)

        mean_ROM, mean_sROM, mean_sROMs = target_function(case)

        for ss in[0, 1, 2]:
            fig, ax = plt.subplots()
            ax.plot(x, X_test[ss, :], "kx--", ms=3, lw=1, markevery=(0, markevery), label=r"$u_e$: exact solution")
            ax.plot(x, X_test_ROM[ss, :], "C0o-", ms=2, lw=1, markevery=(15, markevery), label=r"$u_{rb}$: standard ROM")
            #ax.plot(x, X_test_sROM[ss, :], "C1--", ms=2, lw=1, label=r"$u_{rb, S}$: S-ROM")
            ax.plot(x, X_test_sROMs[ss, :], "C2|-", ms=2, lw=1, markevery=(30, markevery), label=r"$u_{rb, DS}$: DS-ROM")
            # plt.plot(x, X_test[0], "kx-", ms=4, markevery=(0, markevery), label=r"$u_e$: exact solution")
            # plt.plot(x, X_test_sROM[0], "C1--", ms=4, markevery=(15, markevery), label=r"$u_{rb, S}$: S-ROM")
            # plt.plot(x, X_train[1], "C3-3", ms=4, lw=.5, markevery=(30, markevery), label=r"smoothed snapshot \#2")
            # plt.plot(x, X_train[2], "C3-4", ms=4, lw=.5, markevery=(30, markevery), label=r"smoothed snapshot \#3")
            
            ax.set_xticks(np.linspace(0, 1, 11, endpoint=True), minor=True)
            ax.set_yticks(np.linspace(0, 1, 11, endpoint=True), minor=True)
            plt.grid(True, which='minor', linestyle='--', lw=.25)
            plt.grid(True, which='major', linestyle='-')
            plt.legend(loc="upper right")
            if name=="small_amp_pulse":
                plt.ylim(-.0005, .012)
                ax.text(0.999, 0.0035, r"$||u_{rb}-u_e||_{L_2}"+"={:.5f}$".format(e_ROM[ss]), horizontalalignment='right', verticalalignment='center')
                #ax.text(0.999, 0.25, r"$||u_{rb, S}-u_e||_{L_2}"+"={:.2f}$".format(e_sROM[ss]), horizontalalignment='right', verticalalignment='center')
                ax.text(0.999, 0.0025, r"$||u_{rb, DS}-u_e||_{L_2}"+"={:.5f}$".format(e_sROMs[ss]), horizontalalignment='right', verticalalignment='center')
            else:
                ax.text(0.999, 0.35, r"$||u_{rb}-u_e||_{L_2}"+"={:.2f}$".format(e_ROM[ss]), horizontalalignment='right', verticalalignment='center')
                #ax.text(0.999, 0.25, r"$||u_{rb, S}-u_e||_{L_2}"+"={:.2f}$".format(e_sROM[ss]), horizontalalignment='right', verticalalignment='center')
                ax.text(0.999, 0.25, r"$||u_{rb, DS}-u_e||_{L_2}"+"={:.2f}$".format(e_sROMs[ss]), horizontalalignment='right', verticalalignment='center')
            plt.xlabel("x")
            plt.ylabel("y")
            plt.xlim(0, 1)
            pth2 = "../Plots/10_"+name
            plt.savefig(pth2+"_r"+str(rank)+"_ss"+str(ss)+".pdf", bbox_inches='tight')
            plt.show()

    # for i, rank in enumerate(NN):
    #     n_train = rank
    #     print(rank)
    #     case["rank"] = rank
    #     case["n_train"] = n_train
    #     case["sigma"], case["c"] = sS_sroms2[i], c_sroms2[i]
    #     case["n_test"] = 100
    #     mean_ROM, mean_sROM, mean_sROMs = target_function(case)
    #     e_rom[i] = mean_ROM
    # np.save(pth+"_e_rom.npy", e_rom)

In [ ]:
N_max = 500
# for name, ic in zip(["rectangular_pulse","rect_pulse_sin","saw_tooth","triangle","medium_r_pulse","very_small_r_pulse","step","small_saw_tooth"],
#                     [rectangular_pulse, rect_pulse_sin, saw_tooth, triangle, medium_r_pulse, very_small_r_pulse, step, small_saw_tooth]):
for name, ic in zip(["medium_r_pulse", "small_amp_pulse", "rectangular_pulse", "rect_pulse_sin", "saw_tooth", "step", "triangle"], # "small_saw_tooth", "rectangular_pulse", "very_small_r_pulse", 
                    [medium_r_pulse, small_amp_pulse, rectangular_pulse, rect_pulse_sin, saw_tooth, step, triangle]): # , small_saw_tooth, rectangular_pulse, very_small_r_pulse
    case["g"] = ic
    print(name)
    pth = "../data/10_"+name
    NN = np.load(pth+"_NN.npy")
    
    sS_srom = np.load(pth+"_sS_srom.npy")[:len(NN)]
    sS_sroms1 = np.load(pth+"10_sS_sroms1.npy")[:len(NN)]
    sD_sroms1 = np.load(pth+"10_sD_sroms1.npy")[:len(NN)]
    sS_sroms2 = np.load(pth+"10_sS_sroms2.npy")[:len(NN)]
    c_sroms2 = np.load(pth+"10_c_sroms2.npy")[:len(NN)]
    
    e_rom = np.load(pth+"_e_rom.npy")[:len(NN)]
    e_srom = np.load(pth+"_e_srom.npy")[:len(NN)]
    e_sroms1 = np.load(pth+"_e_sroms1.npy")[:len(NN)]
    e_sroms2 = np.load(pth+"_e_sroms2.npy")[:len(NN)]
    
    m = np.dot(NN, 1/sS_sroms2) / np.dot(NN, NN)
    sS_prescr = 1/(m*NN)
    print(m)

    
    case["g"] = ic
    e_dsrom_prescribed_sigma = np.zeros_like(e_sroms2)
    for i, rank in enumerate(NN):
        n_train = rank
        print(rank)
        case["rank"] = rank
        case["n_train"] = n_train
        case["sROM_only"] = False
        case["sigmaD"] = "calc_based_on_distance"
        case["n_test"] = 100
        case["sigma"], case["c"] = sS_prescr[i], 2.0  # c=1 is an arbitrary choice sS_prescr, sS_sroms2
        mean_ROM, mean_sROM, mean_sROMs = target_function(case) 
        e_dsrom_prescribed_sigma[i] = mean_sROMs


    #NN = [10, 20, 30, 50, 75, 100, 125, 150, 175, 200]
    print(name)
    if name=="medium_r_pulse":
        s1, e1 = 1,16
        s2, e2 = 22,41
    else:
        s1, e1 = 0,6
        s2, e2 = 11, 33

    print(NN[s1:e1], NN[s2:e2])
    print("S-ROM         ", 100-np.mean((e_srom/e_rom)[s1:e1])*100, 100-np.mean((e_srom/e_rom)[s2:e2])*100)
    print("DS-ROM, sigmaD", 100-np.mean((e_sroms1/e_rom)[s1:e1])*100, 100-np.mean((e_sroms1/e_rom)[s2:e2])*100)
    print("DS-ROM, alpha ", 100-np.mean((e_sroms2/e_rom)[s1:e1])*100, 100-np.mean((e_sroms2/e_rom)[s2:e2])*100)
    print("DS-ROM, prescribed_sigma", 100-np.mean((e_dsrom_prescribed_sigma/e_rom)[s1:e1])*100)
    pth = "../Plots/10_"+name
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(single_column_in*2, single_column_in/1.618))
    ax1.plot(NN, e_rom, "C0o", ms=3, label=r"ROM")
    ax1.plot(NN, e_srom, "C1o", ms=3, label=r"S-ROM")
    ax1.plot(NN, e_sroms1, "C2o", ms=3, label=r"DS-ROM, $\sigma_D\!=\!\sigma_D(N)$")
    ax1.plot(NN, e_sroms2, "C3o", ms=3, label=r"DS-ROM, $\sigma_D\!=\!\sigma_D(N, \mu)$")  #  (\sigma_S, c)
    ax1.plot(NN, e_dsrom_prescribed_sigma, "k--", ms=3, label=r"DS-ROM, $\sigma_D\!=\!\sigma_D(N, \mu),$"+"\n"+r"$\sigma_S:\!=\!\frac{"+"{:.1f}".format(1/m)+"}{N}$")  #  (\sigma_S, c)
    #plt.plot(NN, dN_ROM, "k.-", ms=2, label="no smoothing")
    # for i in [2, 5, 7]: #range(4, 10):
    #     plt.plot(NN, dN_sig[:, i], ls="-", marker=".", lw=1, ms=1, label=lbls[i])# r"$u_{\mu,Srb}$"
    #plt.plot(NN, dN_sig[:, 5], ls="-", marker=".", lw=1, ms=1, label=r"$\sigma_S=$"+lbls[5])# r"$u_{\mu,Srb}$"
    #plt.plot(NN, dN_sig[:, 0], ls="None", marker="x", lw=1, ms=2, label=r"optimized $\sigma_S$")# r"$u_{\mu,Srb}$"
    #plt.plot(NN, .4/NN**.5, "C2--", lw=1, ms=1, label=r"$0.4/\sqrt{N}$")# r"$u_{\mu,Srb}$"
    ax1.set_ylabel(r"$\|u_{rb}-u_{e}\|_{L_2}$")
    ax1.set_xlabel(r"$N$")
    # plt.ylim(0, 0.4)
    ax1.legend(loc='upper center')
    ax1.set_xticks(np.linspace(0, N_max, 21, endpoint=True), minor=True)
    ax1.set_yticks(np.linspace(0, 0.4, 19, endpoint=True), minor=True)
    ax1.grid(True, which="minor", linestyle="--", lw=0.25)
    ax1.grid(True, which="major", linestyle="-")
    #ax.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_xlim(0, N_max)
    #plt.ylim(1e-2, 0.11)
    # plt.savefig(pth+"error.pdf")
    # plt.show()
    # m, b = np.polyfit(NN, 1/sS_sroms2, deg=1)
    # print(m, b)
    # fig, ax =  plt.subplots()
    ax2.plot(NN, 0.2/NN, "k:", ms=3, label=r"$\sigma_S\!=\!\frac{0.2}{N}$")
    ax2.plot(NN, 1/(m*NN), "k--", ms=3, label=r"$\sigma_S\!=\!\frac{"+"{:.1f}".format(1/m)+"}{N}$")
    ax2.plot(NN, sS_srom, "C1o", ms=3, label=r"$\sigma_S$: S-ROM")
    ax2.plot(NN, sS_sroms1, "C2o", ms=3, label=r"$\sigma_S$: DS-ROM, $\sigma_D\!=\!\sigma_D(N)$")
    #ax.plot(NN, sD_sroms1, "C2x", ms=3, label=r"$\sigma_D: DS-ROM, \sigma_D=\sigma_D(N)$")
    ax2.plot(NN, sS_sroms2, "C3o", ms=3, label=r"$\sigma_S$: DS-ROM, $\sigma_D\!=\!\sigma_D(N, \mu)$")  # (\sigma_S, c)
    #ax2 = ax.twinx()
    #ax2.plot(NN, c_sroms2, "kx", label="c")
    #sS_sroms2[i] = parameters[0]
     #   c_sroms2[i] = parameters[1]
    ax2.set_ylabel(r"$\sigma_S$")
    ax2.set_xlabel(r"$N$")
    ax2.set_xticks(np.linspace(0, N_max, 21, endpoint=True), minor=True)
    ax2.set_yticks(np.linspace(0, 0.4, 19, endpoint=True), minor=True)
    ax2.grid(True, which="minor", linestyle="--", lw=0.25)
    ax2.grid(True, which="major", linestyle="-")
    #ax.set_xscale("log")
    ax2.set_yscale("log")
    ax2.set_xlim(0, N_max)
    ax2.set_ylim(1e-4, 0.11)
    #ax2.legend()
    # plt.savefig(pth+"sigma.pdf")
    # plt.show()
    # fig, ax =  plt.subplots()
    ln1 = ax3.plot(NN, sD_sroms1, "C2x", ms=3, label=r"$\sigma_D$: DS-ROM, $\sigma_D\!=\!\sigma_D(N)$")
    ax32 = ax3.twinx()
    ln2 = ax32.plot(NN, c_sroms2, "C3x", ms=3, label=r"$\alpha$: DS-ROM, $\sigma_D\!=\!\sigma_D(N, \mu)$")
    ax32.set_ylabel(r"$\alpha$")
    ax2.legend() # loc='upper center'
    ax3.set_xlabel(r"$N$")
    # plt.ylim(0, 0.4)
    ax3.set_xticks(np.linspace(0, N_max, 21, endpoint=True), minor=True)
 #   ax.set_yticks(np.linspace(0, 0.4, 19, endpoint=True), minor=True)
    ax3.grid(True, which="minor", linestyle="--", lw=0.25)
    ax3.grid(True, which="major", linestyle="-")
    #ax.set_xscale("log")
    ax3.set_yscale("log")
    ax3.set_xlim(0, N_max)
    ax3.set_ylim(1e-4, 0.11)
    ax32.set_ylim(.5, 2)
    lns = ln1+ln2
    labs = [l.get_label() for l in lns]
    ax32.legend(lns, labs) # , loc='upper center'
    ax3.set_ylabel(r"$\sigma_D$")
    plt.tight_layout(pad=0.0, w_pad=0.5, h_pad=0.0)
    #ax2.legend()
    #plt.savefig(pth+"sigmaD_c.pdf")
    plt.savefig(pth+"_results.pdf")
    plt.show()

    N_max = 200



In [ ]:
pth+"_results.pdf"